In [1]:
%matplotlib inline
%reload_ext autoreload
%autoreload 2

In [2]:
import sys

sys.path.append('../../scripts')

In [3]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os
DATA_ROOT = '/data2/a330d' #os.environ.get("DATA_ROOT", ".")
import numpy as np
import glob
import numpy as np
import json
import matplotlib as mpl
#mpl.rcParams["font.family"] = "monospace"

from plotting import plot_model_comparison

/data/a330d/miniforge3/envs/cellina-graph/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
dataset_name = "crc" # Options: merfish, crc

In [5]:
corr_dir = os.path.join(DATA_ROOT, f"datasets/{dataset_name}/correlations")
pattern = os.path.join(corr_dir, "*.json")
files = sorted(glob.glob(pattern))

rows = []
for fp in files:
    name = os.path.basename(fp)
    core = name[len("crc_"):-len(".json")] if dataset_name == "crc" else name[:-len(".json")]
    parts = core.split("_")
    sid = parts[0]
    model_name = parts[1]
    holdout_celltype = "_".join(parts[2:])
    try:
        with open(fp, "r") as f:
            data = json.load(f)
    except Exception:
        # skip unreadable/invalid json
        continue

    try:
        rows.append({
            "sid": f"crc_{sid}" if dataset_name == "crc" else sid,
            "model_name": model_name,
            "holdout_celltype": holdout_celltype,
            "n_deg": data.get("n_deg"),
            "spearman": data.get("spearman"),
            "pearson": data.get("pearson"),
            "precision": data.get("precision"),
            "direction_match": data.get("direction_match"),
            "direction_match_k": data.get("direction_match_k"),
            "direction_match_gt": data.get("direction_match_gt"),
            "mixing_index": data.get("mixing_index"),
            "edistance_global": data.get("edistance_global"),
            "edistance_local": data.get("edistance_local"),
            "edistance_pca": data.get("edistance_pca"),
            "edistance_pca_log": data.get("edistance_pca_log"),           
            "rmse": data.get("rmse"),
            "mse_lfc": data.get("mse_lfc"),
            "nb_deviance": data.get("nb_deviance"),
        })
    except Exception as e:
        print(f"Error processing file {fp}: {e}")
        continue

data_df = pd.DataFrame(rows)
data_df.head()

,sid,model_name,holdout_celltype,n_deg,spearman,pearson,precision,direction_match,direction_match_k,direction_match_gt,mixing_index,edistance_global,edistance_local,edistance_pca,edistance_pca_log,rmse,mse_lfc,nb_deviance
0,crc_120,baseline-cf,Endothelial_CRC,50,0.571861,0.551004,0.10,1.0,0.10,0.98,0.540591,81.515670,82.624410,362.785612,22.230377,5851.174053,12.837272,NaN
1,crc_120,baseline-cf,Epithelial_CRC,50,0.432701,0.317540,0.20,0.8,0.16,0.64,0.896087,65.666152,65.522427,974.472388,51.507393,650873.943596,33.151102,NaN
2,crc_120,baseline-cf,Fibroblast_CRC,50,0.362689,0.391688,0.16,1.0,0.16,0.64,0.605597,83.278736,83.423279,923.764221,24.394183,179619.295161,11.690496,NaN
3,crc_120,baseline-cf,Myeloid_CRC,50,0.493205,0.481838,0.14,1.0,0.14,0.72,0.504394,81.112156,81.538293,390.493773,22.821356,36229.938262,11.068008,NaN
4,crc_120,baseline-cf,T_cell_CRC,50,0.826939,0.713368,0.12,1.0,0.12,0.96,0.784828,88.202479,91.352711,233.463751,22.375911,22622.722118,18.987500,NaN


In [6]:
# Remove -cf from the end of each model_name
data_df["model_name"] = data_df["model_name"].str.replace("-cf", "", regex=False)
n_deg = data_df["n_deg"].iloc[0]
data_df.head()

,sid,model_name,holdout_celltype,n_deg,spearman,pearson,precision,direction_match,direction_match_k,direction_match_gt,mixing_index,edistance_global,edistance_local,edistance_pca,edistance_pca_log,rmse,mse_lfc,nb_deviance
0,crc_120,baseline,Endothelial_CRC,50,0.571861,0.551004,0.10,1.0,0.10,0.98,0.540591,81.515670,82.624410,362.785612,22.230377,5851.174053,12.837272,NaN
1,crc_120,baseline,Epithelial_CRC,50,0.432701,0.317540,0.20,0.8,0.16,0.64,0.896087,65.666152,65.522427,974.472388,51.507393,650873.943596,33.151102,NaN
2,crc_120,baseline,Fibroblast_CRC,50,0.362689,0.391688,0.16,1.0,0.16,0.64,0.605597,83.278736,83.423279,923.764221,24.394183,179619.295161,11.690496,NaN
3,crc_120,baseline,Myeloid_CRC,50,0.493205,0.481838,0.14,1.0,0.14,0.72,0.504394,81.112156,81.538293,390.493773,22.821356,36229.938262,11.068008,NaN
4,crc_120,baseline,T_cell_CRC,50,0.826939,0.713368,0.12,1.0,0.12,0.96,0.784828,88.202479,91.352711,233.463751,22.375911,22622.722118,18.987500,NaN


In [7]:
df = data_df.copy() # start with existing dataframe
df["sid"] = df["sid"].astype(str)

In [8]:
# Remove model_name 'cellina', 'cellina-ablated', 'cellina-graph'
df = df[~df["model_name"].isin(["cellina", "cellina-ablated", "cellina-graph", "cellina-W-1", "cellina-W-2", "cpa-1", "cpa-2", "scgen-1", "scgen-2", "baseline"])]
df.model_name.unique()

array(['cellina-W', 'cellina-ablated-W', 'cellina-graph-W', 'cpa',
       'scgen'], dtype=object)

In [9]:
# Remove -W from model names
df["model_name"] = df["model_name"].str.replace("-W", "", regex=False)
df.model_name.unique()

array(['cellina', 'cellina-ablated', 'cellina-graph', 'cpa', 'scgen'],
      dtype=object)

In [12]:
# CRC
# Print mean and std for each model_name over 'nb_deviance' column, rounded to 3 decimal places
df.groupby("model_name")["nb_deviance"].agg(["mean", "std"]).sort_values(by="mean", ascending=False).round(3)

,mean,std
model_name,,
scgen,0.087,0.035
cpa,0.042,0.047
cellina-ablated,0.036,0.043
cellina-graph,0.032,0.040
cellina,0.030,0.035


In [ ]:
# MERFISH
df.groupby("model_name")["nb_deviance"].mean().sort_values(ascending=False)

model_name
scgen              0.327819
cellina-ablated    0.088414
cpa                0.052624
cellina            0.041722
cellina-graph      0.035987
Name: nb_deviance, dtype: float64